In [0]:
pip install databricks-labs-dqx


In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule, DQDatasetRule, DQForEachColRule
from databricks.sdk import WorkspaceClient

raw_posts_df = spark.read.table("default.posts")
raw_users_df = spark.read.table("default.users")

dqEngine = DQEngine(WorkspaceClient())
checks = [
  *DQForEachColRule(
    columns=["id","creationDate"],
    criticality="error",
    check_func=check_funcs.is_not_null_and_not_empty,
  ).get_rules(),
  DQRowRule(
      name ="creationDate_not_in_Future",
      criticality ="error",
      check_func = check_funcs.is_not_in_future,
      column ="CreationDate"
  )
]
valid_posts_df, Quarantined_df = dqEngine.apply_checks_and_split(raw_posts_df,checks)
valid_users_df, Quarantined_users_df = dqEngine.apply_checks_and_split(raw_users_df,checks)

In [0]:
valid_posts_df.limit(10).display()


In [0]:
Quarantined_df.limit(10).display()

In [0]:
Quarantined_users_df.limit(10).display()